# 7장 · Git 협업·디버깅과 재현 가능한 Python 과제

학습 기록은 30분, 20분, 40분인데 프로그램은 합계로 40분을 냅니다. 같은 입력으로 증상을 다시 만들고, 고친 줄을 동료에게 어떻게 보여 줄지 실습합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 합계가 마지막 값이 되는 줄 찾기

디버깅은 막연히 코드를 바꾸기 전에 틀린 결과를 같은 입력으로 다시 만드는 일부터 시작합니다. 반복을 한 바퀴 돌 때마다 total에 직전 바퀴까지 더한 값이 남아 있어야 합니다.



**볼 곳**　‘첫 기록’에서 시작해 ‘누적 합계’를 거친 뒤 ‘다음 기록’에서 다시 판단하는 흐름을 따라가세요.

**애니메이션의 의미**　합계를 반복문 안에서 초기화하면 앞선 기록이 사라집니다.


### 셀 01 · 40분만 남은 이유

첫 번째 반복이 끝났을 때의 total을 떠올리고 둘째 반복의 첫 줄을 읽으세요.

In [1]:
records = [{"minutes": 30}, {"minutes": 20}, {"minutes": 40}]
total = 0
for record in records:
    total = 0
    total += record["minutes"]
print(total)

40


40이 나옵니다. 30과 20이 더해지기는 하지만 다음 바퀴마다 다시 0이 됩니다. 마지막 바퀴에서 더한 40만 남습니다.

### 과제 02 · 합계를 고치고 재검사하기

total 초기화 위치를 바꿔 세 건의 합을 돌려주는 함수를 작성하세요. 입력 자료는 그대로 씁니다.

In [2]:
def total_minutes(records):
    pass  # 세 기록의 분을 더해 반환하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def total_minutes(records):
    total = 0
    for record in records:
        total += record[&quot;minutes&quot;]
    return total

assert total_minutes(records) == 90
print(total_minutes(records))</code></pre>
<p>실행 결과</p><pre><code>90</code></pre>
<p>90이면 세 기록의 합이 맞습니다. 40으로 되돌아가면 assert에서 멈추므로 고친 문제의 재발을 바로 찾을 수 있습니다.</p>
</details>

## 동료가 보게 될 변경 한 줄

Git의 diff에서 -로 시작하는 본문 줄은 이전 내용, +는 새 내용입니다. 아래에서는 같은 형식의 차이를 표준 라이브러리로 만들어 읽습니다. 앞뒤 문맥이 없으면 초기화한 위치를 오해하기 쉽습니다.



**볼 곳**　‘변경 한 줄’를 기준으로 양쪽의 ‘이전 코드’와 ‘수정 코드’를 대조하세요.

**애니메이션의 의미**　초기화 줄을 어디로 옮겼는지 비교하세요.


### 셀 03 · 코드 차이에서 삭제한 줄 찾기

잘못된 계산에서 반복문 안쪽 초기화 한 줄을 뺐습니다.

In [3]:
from difflib import unified_diff
before = ["total = 0", "for record in records:", "    total = 0", '    total += record["minutes"]']
after = [before[0], before[1], before[3]]
print("\n".join(unified_diff(before, after, fromfile="이전", tofile="수정", lineterm="")))

--- 이전
+++ 수정
@@ -1,4 +1,3 @@
 total = 0
 for record in records:
-    total = 0
     total += record["minutes"]


마이너스가 붙은 total = 0 줄 앞에 들여쓰기가 있습니다. 바깥의 초기화는 남아 있어 첫 바퀴도 정상적으로 시작합니다.

## 같은 줄을 고친 두 사람의 내용 합치기

서로 다른 브랜치에서 같은 줄을 다르게 고치면 Git이 자동으로 한쪽을 선택할 수 없습니다. 충돌 표시 위쪽과 아래쪽을 읽고 두 사람이 고친 의도를 살려 최종 문장을 씁니다.



**볼 곳**　‘내 실행 안내’와 ‘동료 안내’가 오른쪽의 ‘합친 안내’에 함께 들어가는 위치를 보세요.

**애니메이션의 의미**　두 사람의 안내 중 함께 남겨야 할 내용을 합칩니다.


### 셀 04 · 충돌 표시 읽기

수업용 안내문의 같은 줄에 서로 다른 실행 방법을 적었습니다.

아래 문자열에는 이미 충돌 표시가 들어 있습니다. 이번 셀에서는 양쪽이 바꾼 줄을 찾아보세요. 실제 저장소에서 병합을 마칠 때에는 별도로 파일의 충돌 표시를 정리하고 선택한 내용을 기록합니다.

In [4]:
conflict = """<<<<<<< HEAD
전체 셀을 실행하세요.
=======
새 커널에서 시작하세요.
>>>>>>> partner"""
print(conflict)

<<<<<<< HEAD
전체 셀을 실행하세요.
새 커널에서 시작하세요.
>>>> partner


HEAD 쪽은 전체 실행을, partner 쪽은 새 커널로 시작할 것을 강조합니다. 구분선 자체는 안내문 내용에 넣지 않습니다. 실제 브랜치에서 해결한 파일은 다시 스테이징하고 병합을 마칩니다.

### 과제 05 · 동료가 재현할 실행 안내 만들기

두 안내를 한 문장으로 합치고 30·20·40분의 기대 합계까지 적으세요.

In [5]:
# 충돌 표시 없이 인수인계 문장을 완성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(&quot;새 커널에서 전체 셀을 실행하세요. 30·20·40분의 합은 90분입니다.&quot;)</code></pre>
<p>실행 결과</p><pre><code>새 커널에서 전체 셀을 실행하세요. 30·20·40분의 합은 90분입니다.</code></pre>
<p>동료가 같은 입력과 상태에서 다시 확인할 수 있습니다. pytest로 검사를 묶을 때도 위의 assert처럼 입력과 기대값을 가까이 둡니다.</p>
</details>

## 돌아보기

기록을 한 건만 넣으면 초기화가 잘못된 코드도 우연히 통과합니다. 빈 목록과 서로 다른 분을 가진 두 건으로 새 검사를 만들어 보고, 어떤 입력에서 고치기 전·후가 구별되는지 설명해 보세요.